# Early medium-world curriculum — analysis

This is the **600-update learning-order pilot**, comparing `mixed`, `staged` and `reverse` training. We investigate whether the model begins learning shape and binding in such a short budget. This notebook reads the saved experiment; it does not train anything.

In [ ]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

source = PROJECT_ROOT / 'outputs/medium_world/experiments/learning_order/schedule_comparison.json'
report = json.loads(source.read_text(encoding='utf-8'))
runs = report['runs']
SCHEDULES = ('mixed', 'staged', 'reverse')
TASKS = ('color', 'shape', 'conjunction', 'pair')
print('Updates:', runs[0]['steps_run'], '| Runs:', len(runs), '| LR:', report['config']['lr'])

## 1. Experimental exposure

The pilot used **600 updates**, three presentations of each training example, and a learning rate of `0.002`. Compare the same seeds under the same exposure budget. Later studies have longer budgets and different optimization settings; their scores are not directly attributable to curriculum order.

In [ ]:
exposure = pd.DataFrame([{
    'seed': r['seed'], 'schedule': r['schedule'], 'updates': r['steps_run'],
    'total_presentations': r['exposure_plan']['total_presentations'],
    **r['exposure_plan']['counts_by_template']
} for r in runs])
display(exposure.sort_values(['seed', 'schedule']).reset_index(drop=True))

## 2. Final performance and visual grounding

`pair` asks whether a color and shape occur **on the same object**, rather than merely somewhere in the image. Compare binding accuracy with the constant-image control to see whether the model relies on the actual visual input.

In [ ]:
rows = []
for r in runs:
    f = r['final']
    rows.append({
        'seed': r['seed'], 'schedule': r['schedule'],
        'overall_id': 100 * f['id_dev']['accuracy'],
        'ood_pair': 100 * f['ood_dev']['by_template']['pair'],
        'ood_bce': f['ood_dev']['loss'],
        'constant_image': 100 * f['id_dev_constant_image']['accuracy'],
        **{t: 100 * f['id_dev']['by_template'][t] for t in TASKS}
    })
final = pd.DataFrame(rows)
display(final.sort_values(['seed', 'schedule']).round(2))
display(final.groupby('schedule')[['overall_id', 'shape', 'pair', 'ood_pair', 'constant_image']]
        .agg(['mean', 'std']).reindex(SCHEDULES).round(2))

## 3. Paired comparisons

For each seed \(s\), compute the difference between two schedules trained from matching seeds:

\[\overline{\Delta}=\frac{1}{S}\sum_{s=1}^{S}(A_{s,\mathrm{staged}}-A_{s,\mathrm{mixed}}).\]

Comparing paired differences is more informative than selecting the best seed. With five seeds, small differences remain exploratory.

In [ ]:
for first, second in (('staged', 'mixed'), ('reverse', 'mixed'), ('staged', 'reverse')):
    print(f'\n{first} − {second}')
    for metric in ('pair', 'ood_pair', 'overall_id', 'shape'):
        p = final.pivot(index='seed', columns='schedule', values=metric)
        difference = p[first] - p[second]
        print(f'{metric:12s}: {difference.mean():+6.2f} pp | '
              f'wins {(difference > 0).sum()}/{len(difference)}')

## 4. Learning trajectories

A curriculum may change **when** a skill emerges without changing its final accuracy. The saved pilot records evaluations from step 100 onward, so these plots show only the observed part of the trajectory.

In [ ]:
history = pd.DataFrame([{
    'seed': r['seed'], 'schedule': r['schedule'],
    'step': h['step'], 'phase': h['phase'],
    **{t: 100 * h['id_dev']['by_template'][t] for t in TASKS}
} for r in runs for h in r['history']])
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, task in zip(axes, ('pair', 'shape')):
    for schedule in SCHEDULES:
        curve = history[history.schedule == schedule].groupby('step')[task].agg(['mean', 'std'])
        ax.plot(curve.index, curve['mean'], label=schedule)
        ax.fill_between(curve.index, curve['mean'] - curve['std'],
                        curve['mean'] + curve['std'], alpha=.12)
    ax.set(title=f'ID {task}', xlabel='Optimizer updates', ylabel='Accuracy (%)')
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Interference when the stage changes

We compare the last recorded evaluation of each phase with the first evaluation of the next one. A drop in an earlier task is consistent with **interference**, but it also includes further optimization updates and cannot isolate forgetting causally.

In [ ]:
changes = []
for r in runs:
    for before, after in zip(r['history'], r['history'][1:]):
        if before['phase'] == after['phase']:
            continue
        changes.append({
            'seed': r['seed'], 'schedule': r['schedule'],
            'transition': before['phase'] + ' → ' + after['phase'],
            'updates_between': after['step'] - before['step'],
            **{t: 100 * (after['id_dev']['by_template'][t] -
                         before['id_dev']['by_template'][t]) for t in TASKS}
        })
changes = pd.DataFrame(changes)
display(changes.groupby(['schedule', 'transition'])
        [list(TASKS)].mean().round(2))

## What this pilot can tell us

Inspect whether the model can learn shape and binding at all within 600 updates, and whether reverse staging temporarily disrupts earlier skills. A difference from a later 3,000- or 9,000-update experiment may reflect learning rate, training time, or curriculum; this pilot alone cannot distinguish them.